# RAG with your document
- openai
- pinecone for indexing
- llamaindex

In [1]:
!pip install openai
!pip install pinecone-client[grpc]
!pip install llama-index-vector-stores-pinecone
!pip install llama-index>=0.9.31

In [ ]:
import os
import logging
import sys
from typing import List

import openai
from pinecone import Pinecone, ServerlessSpec
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Document,
    StorageContext,
    Settings,
)
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# Set up logging
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logger = logging.getLogger(__name__)

# Load API keys
OPENAI_API_KEY = userdata.get("OPENAI_KEY")
PINECONE_API_KEY = userdata.get("PINECONE")

if not OPENAI_API_KEY or not PINECONE_API_KEY:
    raise ValueError("Please set the OPENAI_API_KEY and PINECONE_API_KEY environment variables.")

# Initialize OpenAI and Pinecone clients
openai.api_key = OPENAI_API_KEY
pc = Pinecone(api_key=PINECONE_API_KEY)

# Set up LlamaIndex
Settings.embed_model = OpenAIEmbedding()
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
def init_pinecone_index(index_name: str, dimension: int = 1536) -> None:
    """Initialize Pinecone index if it doesn't exist."""
    if index_name not in pc.list_indexes():
        pc.create_index(
            name=index_name,
            dimension=dimension,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )
        logger.info(f"Created new Pinecone index: {index_name}")
    else:
        logger.info(f"Pinecone index {index_name} already exists")

def build_vector_store_index(documents: List[Document], index_name: str) -> VectorStoreIndex:
    """Build or load the VectorStoreIndex."""
    pinecone_index = pc.Index(index_name)
    vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    return VectorStoreIndex.from_documents(
        documents, storage_context=storage_context
    )

def load_and_process_documents(directory: str) -> List[Document]:
    """Load and process documents from a directory."""
    return SimpleDirectoryReader(directory).load_data()

In [ ]:
class RAGQABot:
    def __init__(self, index_name: str, documents_dir: str):
        self.index_name = index_name
        self.documents_dir = documents_dir
        self.index = None
        self.query_engine = None

    def initialize(self):
        """Initialize the RAG QA bot."""
        init_pinecone_index(self.index_name)
        documents = load_and_process_documents(self.documents_dir)
        self.index = build_vector_store_index(documents, self.index_name)
        self.query_engine = self.index.as_query_engine()
        logger.info("RAG QA bot initialized successfully")

    def query(self, question: str) -> str:
        """Process a query and return the response."""
        if not self.query_engine:
            raise ValueError("Query engine not initialized. Call initialize() first.")

        response = self.query_engine.query(question)
        return str(response)

In [ ]:
def main():
    index_name = "business-qa"
    documents_dir = "/content/data/paul_graham/"

    # Initialize the RAG QA bot
    qa_bot = RAGQABot(index_name, documents_dir)
    qa_bot.initialize()

    print("Welcome to the QA bot!")
    print("Type 'exit' to quit.")

    while True:
        question = input("\nEnter your question: ")
        if question.lower() == 'exit':
            break

        answer = qa_bot.query(question)
        print(f"\nAnswer: {answer}")

    print("Thank you for using the QA bot!")

if __name__ == "__main__":
    main()